# Algoritmi Greedy: problema selecției activităților

**Echipa:** Elcu Adrian, Țurlea Ecaterina, Ursache Andrei

**Cuprins**
1. Ce este tehnica Greedy
2. Enunțul problemei
3. Ideea Greedy și de ce este corectă
4. Implementare
5. Rulare live, pas cu pas
6. Teste: cazuri limită și verificare cu forța brută
7. Când NU merge: alte criterii Greedy
8. Când Greedy nu e optim: problema rucsacului 0/1
9. Cât de eficientă este soluția? Se poate mai bine?
10. Concluzii

## 1. Ce este tehnica Greedy

Un algoritm Greedy („lacom”) construiește soluția **pas cu pas**. La fiecare pas face alegerea care pare **cea mai bună în acel moment** și **nu mai revine** asupra ei.

Schema generală:
1. sortăm elementele după un **criteriu**;
2. le parcurgem o singură dată;
3. păstrăm un element doar dacă se potrivește cu ce am ales deja.

Greedy dă soluția optimă doar dacă problema are două proprietăți:
- **proprietatea alegerii Greedy:** există o soluție optimă care conține prima alegere făcută de Greedy;
- **substructura optimă:** după această alegere, ce rămâne este o problemă de același tip, dar mai mică.

Dacă aceste proprietăți lipsesc, Greedy poate da un rezultat greșit (secțiunile 7 și 8).

## 2. Enunțul problemei

Se dau **n activități** care trebuie să aibă loc în aceeași sală. Fiecare activitate are un nume, o oră de **start** și o oră de **final** (start < final). În sală poate avea loc o singură activitate o dată.

Două activități sunt **compatibile** dacă nu se suprapun. Lucrăm cu intervale de forma **[start, final)**: dacă o activitate se termină la ora 5 și alta începe la ora 5, ele sunt compatibile.

**Cerință:** alegeți un număr **maxim** de activități compatibile două câte două.

- **Date de intrare:** o listă de tupluri `(nume, start, final)`.
- **Date de ieșire:** lista activităților alese, în ordinea în care au loc.

**Exemplu mic:**

| Activitate | Interval |
|---|---|
| Curs | [8, 10) |
| Laborator | [9, 11) |
| Seminar | [10, 12) |

Răspuns: **Curs, Seminar** (2 activități). Laboratorul se suprapune cu amândouă.

## 3. Ideea Greedy și de ce este corectă

**Criteriul:** la fiecare pas alegem activitatea compatibilă care **se termină cel mai devreme**.

Intuiția: cu cât activitatea aleasă se termină mai repede, cu atât rămâne mai mult timp liber pentru celelalte.

**Pașii algoritmului:**
1. Sortăm activitățile crescător după **final**.
2. Luăm prima activitate din lista sortată.
3. Parcurgem restul: acceptăm o activitate doar dacă **start ≥ finalul ultimei activități alese**. Altfel, o sărim.

**De ce este corect (argumentul schimbului):**
- Fie *g* activitatea cu cel mai mic final. Fie *O* o soluție optimă oarecare și *x* activitatea din *O* care se termină prima.
- Avem final(g) ≤ final(x). Toate celelalte activități din *O* încep după final(x), deci și după final(g).
- Înlocuim *x* cu *g* în *O*: nu apare nicio suprapunere, iar numărul de activități rămâne același. Deci există o soluție optimă care îl conține pe *g*: alegerea lui *g* e „sigură”.
- Ce rămâne (activitățile care încep după final(g)) este aceeași problemă, mai mică, deci raționamentul se repetă.

### Date de intrare

In [ ]:
activitati = [
    ('E', 8, 9),
    ('C', 0, 6),
    ('A', 1, 4),
    ('F', 5, 9),
    ('B', 3, 5),
    ('D', 5, 7),
]

print('Date de intrare:')
for nume, start, final in activitati:
    print(f'{nume}: [{start}, {final})')

## 4. Implementare

In [ ]:
def selecteaza_activitati(activitati, afiseaza=False):
    # Pasul 1: sortam crescator dupa final (pozitia 2 din tuplu)
    ordonate = sorted(activitati, key=lambda a: a[2])

    selectate = []
    ultimul_final = None    # inca nu am ales nicio activitate

    # Pasul 2: parcurgem o singura data lista sortata
    for nume, start, final in ordonate:
        compatibila = ultimul_final is None or start >= ultimul_final

        if afiseaza:
            if compatibila:
                print(f'{nume} [{start}, {final}) -> ACCEPTATA')
            else:
                print(f'{nume} [{start}, {final}) -> respinsa: incepe la {start}, '
                      f'dar ultima aleasa se termina la {ultimul_final}')

        if compatibila:
            selectate.append((nume, start, final))
            ultimul_final = final

    return selectate

### Linia timpului

In [ ]:
def deseneaza(activitati, selectate):
    final_maxim = max(a[2] for a in activitati)
    latime = max(len(a[0]) for a in activitati) + 1   # spatiu pentru cel mai lung nume
    print(' ' * latime + ''.join(str(t % 10) for t in range(final_maxim + 1)))

    for nume, start, final in sorted(activitati, key=lambda a: a[2]):
        if (nume, start, final) in selectate:
            simbol = '#'
        else:
            simbol = '-'
        print(nume.ljust(latime) + ' ' * start + simbol * (final - start))
    print('(# = aleasa, - = respinsa)')

## 5. Rulare pas cu pas

In [ ]:
selectate = selecteaza_activitati(activitati, afiseaza=True)

print()
print('Activitati selectate:', [a[0] for a in selectate])
print('Numar de activitati:', len(selectate))
print()
deseneaza(activitati, selectate)

### Date introduse de la tastatură

In [ ]:
activitati_noi = []
while True:
    linie = input('nume start final (Enter gol = stop): ')
    if linie == '':
        break
    nume, start, final = linie.split()
    start, final = int(start), int(final)
    if start >= final:
        print('  Startul trebuie sa fie mai mic decat finalul. Mai incercati.')
        continue
    activitati_noi.append((nume, start, final))

if activitati_noi:
    rezultat = selecteaza_activitati(activitati_noi, afiseaza=True)
    print()
    print('Activitati selectate:', [a[0] for a in rezultat])
    deseneaza(activitati_noi, rezultat)

## 6. Teste

### 6.1 Cazuri limită

In [ ]:
teste = [
    ('lista goala',          [],                                        0),
    ('o singura activitate', [('X', 2, 5)],                             1),
    ('toate se suprapun',    [('X', 1, 10), ('Y', 2, 9), ('Z', 3, 8)],  1),
    ('se ating la capete',   [('X', 1, 3), ('Y', 3, 5), ('Z', 5, 7)],   3),
    ('final egal',           [('X', 1, 5), ('Y', 3, 5), ('Z', 5, 6)],   2),
    ('activitati identice',  [('X', 1, 3), ('Y', 1, 3)],                1),
]

for descriere, date, asteptat in teste:
    obtinut = len(selecteaza_activitati(date))
    stare = 'OK' if obtinut == asteptat else 'GRESIT'
    print(f'{descriere}: asteptat {asteptat}, obtinut {obtinut} -> {stare}')

### 6.2 Verificare cu forța brută

In [ ]:
def sunt_compatibile(activitati):
    # sortam dupa start si comparam fiecare activitate cu urmatoarea
    ordonate = sorted(activitati, key=lambda a: a[1])
    for i in range(1, len(ordonate)):
        if ordonate[i - 1][2] > ordonate[i][1]:
            return False
    return True


def toate_submultimile(elemente):
    # la fiecare element nou, fiecare submultime veche apare o data fara el si o data cu el
    submultimi = [[]]
    for e in elemente:
        submultimi = submultimi + [s + [e] for s in submultimi]
    return submultimi


def maxim_forta_bruta(activitati):
    maxim = 0
    for s in toate_submultimile(activitati):
        if sunt_compatibile(s) and len(s) > maxim:
            maxim = len(s)
    return maxim


print('Activitatile alese nu se suprapun:', sunt_compatibile(selectate))
print('Greedy a ales:', len(selectate), '| maximul posibil (forta bruta):', maxim_forta_bruta(activitati))

### 6.3 Teste aleatoare

In [ ]:
import random

random.seed(1)
corecte = 0
numar_teste = 300

for test in range(numar_teste):
    n = random.randint(0, 10)
    date = []
    for i in range(n):
        start = random.randint(0, 20)
        durata = random.randint(1, 6)
        date.append((f'T{i}', start, start + durata))

    if len(selecteaza_activitati(date)) == maxim_forta_bruta(date):
        corecte += 1

print(f'{corecte} din {numar_teste} teste: Greedy a gasit de fiecare data maximul.')

## 7. Când NU merge: alte criterii Greedy

In [ ]:
def greedy_dupa_criteriu(activitati, criteriu):
    ordonate = sorted(activitati, key=criteriu)
    alese = []
    for activitate in ordonate:
        if sunt_compatibile(alese + [activitate]):
            alese.append(activitate)
    return alese


criterii = [
    ('start cel mai devreme', lambda a: a[1]),
    ('durata cea mai mica',   lambda a: a[2] - a[1]),
    ('final cel mai devreme', lambda a: a[2]),
]


def compara_criterii(activitati):
    for nume_criteriu, criteriu in criterii:
        alese = greedy_dupa_criteriu(activitati, criteriu)
        print(f'{nume_criteriu}: {[a[0] for a in alese]} (total: {len(alese)})')

### 7.1 Criteriul „start cel mai devreme”

In [ ]:
exemplu_start = [
    ('A', 1, 10),
    ('B', 2, 3),
    ('C', 3, 4),
    ('D', 4, 5),
]

compara_criterii(exemplu_start)

### 7.2 Criteriul „durata cea mai mică”

In [ ]:
exemplu_durata = [
    ('A', 0, 4),
    ('B', 5, 9),
    ('S', 3, 6),
]

compara_criterii(exemplu_durata)

## 8. Când Greedy nu e optim: problema rucsacului 0/1

**Enunț:** se dau n obiecte, fiecare cu o greutate și o valoare, și un rucsac care suportă o greutate maximă (capacitatea). Fiecare obiect se ia **întreg sau deloc**. Alegeți obiectele astfel încât greutatea totală să nu depășească capacitatea, iar valoarea totală să fie **maximă**.

**Criteriul Greedy:** raportul valoare/greutate, descrescător.

In [ ]:
obiecte = [
    ('A', 10, 60),
    ('B', 20, 100),
    ('C', 30, 120),
]
capacitate = 50

for nume, greutate, valoare in obiecte:
    print(f'{nume}: greutate={greutate}, valoare={valoare}, raport={valoare / greutate:.1f}')
print('Capacitate:', capacitate)

In [ ]:
def rucsac_greedy(obiecte, capacitate):
    ordonate = sorted(obiecte, key=lambda o: o[2] / o[1], reverse=True)
    alese = []
    greutate_folosita = 0
    for nume, greutate, valoare in ordonate:
        if greutate_folosita + greutate <= capacitate:
            alese.append((nume, greutate, valoare))
            greutate_folosita += greutate
    return alese


def rucsac_forta_bruta(obiecte, capacitate):
    cea_mai_buna = []
    for s in toate_submultimile(obiecte):
        greutate = sum(o[1] for o in s)
        valoare = sum(o[2] for o in s)
        if greutate <= capacitate and valoare > sum(o[2] for o in cea_mai_buna):
            cea_mai_buna = s
    return cea_mai_buna


def descrie(alese):
    nume = [o[0] for o in alese]
    return f'{nume} | greutate = {sum(o[1] for o in alese)} | valoare = {sum(o[2] for o in alese)}'


print('Greedy:', descrie(rucsac_greedy(obiecte, capacitate)))
print('Optim: ', descrie(rucsac_forta_bruta(obiecte, capacitate)))

### 8.1 Rucsacul fracționar

Aceeași problemă, dar obiectele se pot lua și **parțial**.

In [ ]:
def rucsac_fractionar(obiecte, capacitate):
    ordonate = sorted(obiecte, key=lambda o: o[2] / o[1], reverse=True)
    loc_liber = capacitate
    valoare_totala = 0
    for nume, greutate, valoare in ordonate:
        if loc_liber == 0:
            break
        luat = min(greutate, loc_liber)
        valoare_totala += valoare * luat / greutate
        loc_liber -= luat
        print(f'iau {luat} din {greutate} din {nume}')
    return valoare_totala


print('Valoare maxima (fractionar):', rucsac_fractionar(obiecte, capacitate))

## 9. Cât de eficientă este soluția? Se poate mai bine?

| Etapă | Timp |
|---|---|
| Sortarea după final | O(n log n) |
| Parcurgerea listei sortate | O(n) |
| **Total** | **O(n log n)** |

Memorie suplimentară: **O(n)** (lista sortată și lista rezultat).

**Se poate mai bine?**
- Dacă activitățile vin **deja sortate după final**, sărim peste sortare: **O(n)**. Mai repede nu se poate, pentru că fiecare activitate trebuie analizată măcar o dată.
- Dacă activitățile au și o **valoare** și vrem valoarea totală maximă (nu numărul maxim), criteriul Greedy nu mai garantează optimul. Varianta aceasta se rezolvă cu **programare dinamică**.
- La **rucsacul 0/1**, soluția exactă se obține tot cu programare dinamică, în O(n · capacitate).

### Timp de execuție pe date mari

In [ ]:
import time

for n in [10_000, 100_000, 1_000_000]:
    date = []
    for i in range(n):
        start = random.randint(0, 10 * n)
        date.append((i, start, start + random.randint(1, 100)))

    inceput = time.time()
    rezultat = selecteaza_activitati(date)
    durata = time.time() - inceput

    print(f'n = {n:>9}: {durata:.3f} secunde, {len(rezultat)} activitati alese')

## 10. Concluzii

- Pentru selecția activităților, criteriul **„final cel mai devreme”** dă mereu soluția optimă, în **O(n log n)**.
- Criterii care par la fel de naturale (start minim, durată minimă) pot da rezultate greșite.
- La rucsacul 0/1, Greedy după raport nu garantează optimul; la rucsacul fracționar, da.
- Un algoritm Greedy trebuie **demonstrat** pentru fiecare problemă, nu doar intuit.